tensorboard --logdir "variable_leverage_model_recurrent/ppo_mixed"

### 1. **rollout/ep\_rew\_mean**

* **What it is:** The average total reward per episode during training (mean over episodes in the rollout batch).
* **What to look for:**

  * Increasing curve → your agent is learning to get higher rewards.
  * Flat or decreasing → no improvement or problems.
  * Noise is normal but overall upward trend is good.

---

### 2. **rollout/ep\_len\_mean**

* **What it is:** Average episode length during training.
* **What to look for:**

  * Longer episodes usually mean the agent is surviving/acting well.
  * Sudden drops may mean early termination or failures.
  * Combined with rewards, it helps interpret agent stability.

---

### 3. **train/policy\_gradient\_loss**

* **What it is:** The surrogate loss the PPO policy tries to minimize.
* **What to look for:**

  * Generally should decrease and stabilize near zero or a small negative number.
  * Large spikes can indicate instability or big policy updates.
  * Values near zero (small magnitude) often mean stable learning.

---

### 4. **train/value\_loss**

* **What it is:** Loss of the value function (critic) predicting returns.
* **What to look for:**

  * Should decrease steadily over time.
  * Large or increasing values indicate the critic is struggling to predict rewards.
  * If it stays high, consider reward scaling or larger critic network.

---

### 5. **train/entropy\_loss**

* **What it is:** Encourages exploration by measuring randomness in the policy.
* **What to look for:**

  * Negative values; less negative means higher entropy (more exploration).
  * If it rapidly goes very negative (low entropy), policy is becoming too deterministic too early (possible premature convergence).

---

### 6. **train/explained\_variance**

* **What it is:** How well the value function explains the variance in returns.
* **What to look for:**

  * Ranges roughly between -∞ and 1.
  * Values near 1 mean excellent fit (value predicts returns well).
  * Values near 0 mean the value function is no better than the mean return.
  * Negative values indicate very poor prediction (critic may need tuning).

---

### 7. **train/clip\_fraction**

* **What it is:** Fraction of policy updates where the PPO clipping was active (policy changed beyond clip threshold).
* **What to look for:**

  * Values too high (>0.3–0.4) mean PPO updates are often hitting the clip limit — might be too aggressive learning rate.
  * Too low values might mean policy changes are very small, slowing learning.

---

### 8. **train/approx\_kl**

* **What it is:** Approximate KL divergence between old and new policy after update.
* **What to look for:**

  * Keep below 0.03 to 0.05 usually (PPO recommends).
  * Higher values indicate big policy jumps, which may destabilize learning.

---

### Bonus: **Eval/mean\_reward** (if you use `EvalCallback`)

* **What it is:** Mean reward over evaluation episodes with deterministic policy.
* **What to look for:**

  * Should increase steadily, showing your agent is improving on held-out envs.
  * Plateaus or decreases indicate learning stalls or overfitting.

---

# Summary

* **Ideal training run**:

  * `ep_rew_mean` goes up
  * `value_loss` and `policy_gradient_loss` go down and stabilize
  * `explained_variance` approaches 1 or at least positive values
  * `clip_fraction` moderate (\~0.1-0.3)
  * `approx_kl` stays under 0.03
  * `entropy_loss` slowly decreases (less exploration) but not zero

If you see any metrics behaving wildly or flatlining, that’s a signal to tune learning rate, normalization, or network architecture.


### 1. **`ep_rew_mean` or `mean_reward` stuck low / no improvement / drops**

* **What it means:** The agent isn’t learning meaningful policies or is unstable.
* **What to try:**

  * Check environment correctness and rewards (no bugs or extreme sparsity).
  * Increase training timesteps.
  * Lower learning rate (try 1e-4 or 5e-5).
  * Add or increase entropy coefficient (`ent_coef`) to encourage exploration.
  * Normalize observations and rewards (`VecNormalize`).
  * Use larger or deeper policy/value networks.
  * Use curriculum learning or simpler tasks first.

---

### 2. **`explained_variance` near 0 or negative (value function not learning)**

* **What it means:** Critic is not predicting returns well.
* **What to try:**

  * Normalize rewards and observations.
  * Increase value network size.
  * Increase number of epochs per update (`n_epochs`).
  * Reduce learning rate for the value network.
  * Check if reward scale is very large or small and normalize it.
  * Verify advantage calculation and discount factor (`gamma`) correctness.

---

### 3. **`policy_gradient_loss` very noisy or large positive values**

* **What it means:** Policy updates might be unstable or not improving.
* **What to try:**

  * Reduce learning rate.
  * Decrease `clip_range` (default 0.2) to limit policy updates.
  * Increase batch size (`n_steps * n_envs`) for more stable gradient estimates.
  * Increase `n_epochs` for more thorough updates per batch.
  * Try different policy architectures.
  * Ensure advantage estimates are accurate (check value function).

---

### 4. **`value_loss` very large or exploding**

* **What it means:** Critic loss is not converging or diverging.
* **What to try:**

  * Reduce learning rate for value network.
  * Normalize rewards.
  * Use gradient clipping.
  * Increase value network capacity.
  * Check for bugs in reward or return calculation.

---

### 5. **`entropy_loss` very low (large negative) and decreasing**

* **What it means:** Policy is becoming too deterministic too quickly (low exploration).
* **What to try:**

  * Increase entropy coefficient (`ent_coef`) to encourage exploration.
  * Use a learning rate schedule to slow early convergence.
  * Check that action space and policy distribution are correct.

---

### 6. **`clip_fraction` very high (close to 1) or erratic**

* **What it means:** PPO clipping is active almost all the time — policy updates are too large.
* **What to try:**

  * Reduce learning rate.
  * Reduce `clip_range`.
  * Increase batch size (`n_steps * n_envs`).
  * Check for bugs in action computation or environment steps.

---

### 7. **`approx_kl` spikes or is consistently above 0.03-0.05**

* **What it means:** Policy is changing too rapidly, risking instability.
* **What to try:**

  * Lower learning rate.
  * Decrease `clip_range`.
  * Increase batch size.
  * Use early stopping based on KL divergence (`target_kl` in PPO).

---

### 8. **`ep_len_mean` very low or erratic**

* **What it means:** Agent may be failing early or environment episodes are ending prematurely.
* **What to try:**

  * Check environment termination conditions.
  * Tune reward shaping.
  * Increase max episode length if appropriate.
  * Ensure agent is learning to survive longer.

---

### General tips:

* Use **tensorboard logs** to track effects of each change over multiple runs.
* Start with **small, interpretable changes**.
* Validate environment correctness separately before heavy training.
* Make sure vectorized environments and normalization wrappers are applied correctly and consistently.
* Consider **grid or random search** over key hyperparameters like learning rate, entropy coef, batch size, and network size.